# Inference — Generate Answers from a Trained Global Adapter

Decodes free-form answers from a trained global LoRA adapter, given the query and its relevant snippet. Produces the generations that the judge and multiple-choice metrics then score. No loss is computed here.

---

*Notation:* `q` query · `c+` relevant snippet · `c-` off-topic snippet from the same user · `y+` personalised answer · `y-` general answer


In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip -q install -U datasets huggingface_hub transformers accelerate peft bitsandbytes

    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(".")

if IN_COLAB:
    PP_ROOT = DRIVE_ROOT / "privacy_perserving_pllm"
else:
    _here = Path(".").resolve()
    PP_ROOT = next(
        (
            p for p in [_here, *_here.parents]
            if (p / "v2_personamem_persona_subsets").exists()
            or all((p / d).exists() for d in ("centralised", "evaluation", "fed_grad_avg", "zero_shot"))
        ),
        _here,
    )
SUBSETS_DIR = PP_ROOT / "v2_personamem_persona_subsets"

SUBSET_NAMES = ["all_subset"]

METHOD_KEYS = [
    "centralized_sft_snippet",
    "centralized_orpo_snippet",
    "centralized_ctx_contrast_orpo_snippet",
    "centralized_combined_orpo_snippet",
    "centralized_gen_edit_snippet",
    "federated_sft_gradavg_snippet",
    "federated_orpo_gradavg_snippet",
    "federated_ctx_contrast_orpo_gradavg_snippet",
    "federated_combined_orpo_gradavg_snippet",
    "federated_gen_edit_gradavg_snippet"
]

def build_results_dir(subset, method_key):
    """Resolve the results/adapter dir for a (subset, method) pair.

    5-set FedSGD methods live under fed_grad_avg/fed_grad_avg_5_set/<subset>/;
    everything else (incl. zero-shot) lives directly under <subset>/.
    """
    dirname = f"v2_personamem_{method_key}"
    if "5set" in method_key:
        base = PP_ROOT / "fed_grad_avg" / "fed_grad_avg_5_set" / subset
    else:
        base = PP_ROOT / subset
    return base / dirname

ADAPTER_EPOCH = 3
EVAL_BATCH_SIZE = 32

print("Subsets:", SUBSET_NAMES)
print("Methods:", len(METHOD_KEYS))
print("Adapter epoch:", ADAPTER_EPOCH if ADAPTER_EPOCH is not None else "final")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 236.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 63.8 MB/s eta 0:00:00
Mounted at /content/drive
Subsets: ['all_subset']
Methods: 7
Adapter epoch: 3


In [2]:
import ast
import json
import random
from typing import Any, Dict, List

import pandas as pd
from datasets import load_dataset

DATASET_NAME = "bowen-upenn/PersonaMem-v2"
CONFIG = "benchmark"
TEXT_SPLITS = ("train_text", "val_text", "benchmark_text")
SEED = 42
random.seed(SEED)


def load_split(split):
    if split not in TEXT_SPLITS:
        raise ValueError(f"split must be one of {TEXT_SPLITS}, got {split!r}")
    return load_dataset(DATASET_NAME, CONFIG, split=split)


def parse_user_query(raw):
    if isinstance(raw, dict):
        return str(raw.get("content", raw)).strip()
    if isinstance(raw, str):
        try:
            d = ast.literal_eval(raw)
            if isinstance(d, dict):
                return str(d.get("content", raw)).strip()
        except (ValueError, SyntaxError):
            pass
        return raw.strip()
    return str(raw).strip()


def parse_incorrect_answers(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if hasattr(raw, "tolist"):
        return [str(x) for x in raw.tolist()]
    if isinstance(raw, str):
        try:
            val = ast.literal_eval(raw)
            if isinstance(val, list):
                return [str(x) for x in val]
        except (ValueError, SyntaxError):
            pass
        return [raw]
    return []


def get_snippet(row):
    val = row.get("related_conversation_snippet")
    if val is None:
        return ""
    return str(val).strip()

In [3]:
def load_subset_val(name, subsets_dir=SUBSETS_DIR):
    """Return (persona_ids, val_df) for a subset. Only val is needed for inference."""
    subset_dir = Path(subsets_dir) / name
    if not subset_dir.exists():
        raise FileNotFoundError(f"Subset not found: {subset_dir}. Run create_persona_subsets.ipynb first.")
    with open(subset_dir / "personas.json", "r", encoding="utf-8") as f:
        persona_ids = [int(p) for p in json.load(f)]
    va = pd.read_parquet(subset_dir / "val.parquet")
    return sorted(persona_ids), va

In [4]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_SNIPPET_TOKENS = 4096
MAX_NEW_TOKENS = 512

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token
model_tok.padding_side = "left"


def build_system_prompt():
    return (
        "You are a personalised assistant. Use the conversation snippet to find "
        "information or connections relevant to the question, then provide the answer."
    )


def truncate_snippet(snippet, max_tokens):
    ids = model_tok(snippet, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return snippet
    return model_tok.decode(ids[-max_tokens:], skip_special_tokens=True)


def build_user_content(row):
    snippet = truncate_snippet(get_snippet(row), MAX_SNIPPET_TOKENS)
    return (
        "Relevant snippet from our earlier conversation:\n"
        f"{snippet}\n\n"
        f"{parse_user_query(row['user_query'])}"
    )


def build_prompt_text(system_prompt, row):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_user_content(row)},
    ]
    return model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def resolve_adapter_dir(results_dir, epoch=None):
    if epoch is not None:
        return Path(results_dir) / "global" / f"adapter_epoch_{epoch}"
    return Path(results_dir) / "global" / "adapter"


def load_infer_model(results_dir, use_base_only, adapter_epoch=None):
    """Load a fresh base model (+ adapter unless zero-shot). Returns the model."""
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    base.config.use_cache = True

    if use_base_only:
        print("  Using base model only (zero-shot).")
        base.eval()
        return base

    adapter_dir = resolve_adapter_dir(results_dir, adapter_epoch)
    if not adapter_dir.exists():
        raise FileNotFoundError(f"Adapter not found: {adapter_dir}")
    print("  Loading adapter from:", adapter_dir.resolve())
    m = PeftModel.from_pretrained(base, str(adapter_dir))
    m.eval()
    return m


def free_model(model):
    """Delete the model and reclaim GPU memory before loading the next one."""
    try:
        del model
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [5]:
from tqdm.auto import tqdm

EVAL_BATCH_MAX_INPUT_TOKENS = 12_000

def _prompt_token_lengths(prompts):
    enc = model_tok(prompts, add_special_tokens=False)
    return [len(ids) for ids in enc["input_ids"]]

def _make_generation_batches(lengths, *, batch_size, max_input_tokens):
    order = sorted(range(len(lengths)), key=lambda i: lengths[i])
    batches: List[List[int]] = []
    cur: List[int] = []
    cur_max = 0
    for i in order:
        plen = lengths[i]
        new_max = max(cur_max, plen)
        new_total = new_max * (len(cur) + 1)
        if cur and (len(cur) >= batch_size or new_total > max_input_tokens):
            batches.append(cur)
            cur = [i]
            cur_max = plen
        else:
            cur.append(i)
            cur_max = new_max
    if cur:
        batches.append(cur)
    return batches

@torch.no_grad()
def batch_generate_prompts(model, prompts, *, batch_size=EVAL_BATCH_SIZE, max_input_tokens=EVAL_BATCH_MAX_INPUT_TOKENS, desc="generate"):
    if not prompts:
        return []

    lengths = _prompt_token_lengths(prompts)
    batches = _make_generation_batches(
        lengths, batch_size=batch_size, max_input_tokens=max_input_tokens
    )
    gens: List[str] = [""] * len(prompts)

    for batch_idx in tqdm(batches, desc=desc):
        batch_prompts = [prompts[i] for i in batch_idx]
        inputs = model_tok(
            batch_prompts,
            return_tensors="pt",
            add_special_tokens=False,
            padding=True,
        ).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=model_tok.eos_token_id,
        )
        prompt_len = inputs["input_ids"].shape[1]
        new_tokens = out[:, prompt_len:]
        for j, i in enumerate(batch_idx):
            gens[i] = model_tok.decode(new_tokens[j], skip_special_tokens=True).strip()
        del inputs, out, new_tokens
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return gens

def build_prompts(system_prompt, rows):
    return [build_prompt_text(system_prompt, row) for row in rows]

def generation_records(rows, gens):
    """Attach generations + reference fields needed later for ROUGE (no scoring here)."""
    records = []
    for row, gen in zip(rows, gens):
        incorrects = parse_incorrect_answers(row.get("incorrect_answers"))
        rec = dict(row)
        rec["generated_answer"] = gen
        rec["incorrect_answer"] = incorrects[0] if incorrects else ""
        records.append(rec)
    return pd.DataFrame(records)

def infer_split_batched(model, df, system_prompt, client_personas, desc, *, batch_size=EVAL_BATCH_SIZE):
    eval_df = df[df["persona_id"].isin(client_personas)].reset_index(drop=True)
    rows = eval_df.to_dict("records")
    if not rows:
        return pd.DataFrame()
    prompts = build_prompts(system_prompt, rows)
    gens = batch_generate_prompts(model, prompts, batch_size=batch_size, desc=desc)
    return generation_records(rows, gens)

def save_inference_predictions(preds_df, results_dir, split_name, *, method_key, subset, adapter_epoch, use_base_only):
    """Save all-row + per-client generation CSVs (no metrics)."""
    results_dir = Path(results_dir)
    out_dir = results_dir / f"epoch_{adapter_epoch}" if adapter_epoch is not None else results_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    all_path = out_dir / f"all_{split_name}_predictions_run2.csv"
    preds_df.to_csv(all_path, index=False)

    for pid, group in preds_df.groupby("persona_id", sort=True):
        persona_dir = out_dir / f"persona_{int(pid)}"
        persona_dir.mkdir(parents=True, exist_ok=True)
        group.to_csv(persona_dir / f"{split_name}_predictions.csv", index=False)

    meta = {
        "method_key": method_key,
        "subset": subset,
        "split": split_name,
        "n_rows": int(len(preds_df)),
        "num_personas": int(preds_df["persona_id"].nunique()) if len(preds_df) else 0,
        "adapter_epoch": adapter_epoch,
        "use_base_model_only": use_base_only,
        "output_dir": str(out_dir),
        "all_predictions_csv": str(all_path),
    }
    with open(out_dir / f"inference_{split_name}_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    return all_path

In [6]:
system_prompt = build_system_prompt()
print("=== Multi-subset / multi-method inference (val only) ===")
print(f"subsets={SUBSET_NAMES}")
print(f"methods={len(METHOD_KEYS)} | batch_size={EVAL_BATCH_SIZE}")

saved_paths = []
skipped = []

for subset in SUBSET_NAMES:
    personas, val_df = load_subset_val(subset)
    n_val = int(val_df["persona_id"].isin(personas).sum())
    print(f"\n===== SUBSET {subset}: {len(personas)} personas | val rows={n_val} =====")

    for method_key in METHOD_KEYS:
        results_dir = build_results_dir(subset, method_key)
        use_base_only = method_key.startswith("zero_shot")
        print(f"\n--- {subset} / {method_key} ---")

        if not use_base_only:
            adapter_dir = resolve_adapter_dir(results_dir, ADAPTER_EPOCH)
            if not adapter_dir.exists():
                print(f"  [skip] adapter not found: {adapter_dir}")
                skipped.append((subset, method_key))
                continue

        results_dir.mkdir(parents=True, exist_ok=True)
        model = load_infer_model(results_dir, use_base_only, ADAPTER_EPOCH)
        try:
            val_preds = infer_split_batched(
                model, val_df, system_prompt, personas,
                desc=f"{subset}/{method_key} val",
            )
            path = save_inference_predictions(
                val_preds, results_dir, "val",
                method_key=method_key, subset=subset,
                adapter_epoch=ADAPTER_EPOCH, use_base_only=use_base_only,
            )
            saved_paths.append(path)
            print("  saved:", path)
        finally:
            free_model(model)

print("\n=== Done ===")
print(f"saved {len(saved_paths)} prediction files; skipped {len(skipped)}")
for p in saved_paths:
    print("  ", p)
if skipped:
    print("skipped (adapter missing):")
    for s, m in skipped:
        print(f"  {s} / {m}")

=== Multi-subset / multi-method inference (val only) ===
subsets=['all_subset']
methods=7 | batch_size=32

===== SUBSET all_subset: 150 personas | val rows=714 =====

--- all_subset / centralized_sft_snippet ---


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

...

  saved: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_orpo_gradavg_snippet/epoch_3/all_val_predictions_run2.csv

--- all_subset / federated_ctx_contrast_orpo_gradavg_snippet ---


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  Loading adapter from: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_ctx_contrast_orpo_gradavg_snippet/global/adapter_epoch_3


all_subset/federated_ctx_contrast_orpo_gradavg_snippet val:   0%|          | 0/33 [00:00<?, ?it/s]

  saved: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_ctx_contrast_orpo_gradavg_snippet/epoch_3/all_val_predictions_run2.csv

--- all_subset / federated_combined_orpo_gradavg_snippetfederated_gen_edit_gradavg_snippet ---
  [skip] adapter not found: /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_federated_combined_orpo_gradavg_snippetfederated_gen_edit_gradavg_snippet/global/adapter_epoch_3

=== Done ===
saved 5 prediction files; skipped 2
   /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_centralized_sft_snippet/epoch_3/all_val_predictions_run2.csv
   /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_centralized_orpo_snippet/epoch_3/all_val_predictions_run2.csv
   /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personamem_centralized_ctx_contrast_orpo_snippet/epoch_3/all_val_predictions_run2.csv
   /content/drive/MyDrive/privacy_perserving_pllm/all_subset/v2_personam